# 10 — Confronto tra detector (training + valutazione)

**Esperimento 1 della tesi.** Confronto di più detector sullo stesso dataset e con lo stesso protocollo.

| Modello | Ruolo |
|---|---|
| YOLO11s pre-addestrato COCO (classe `person`) | baseline senza dominio basket |
| YOLO11n / YOLO11s / YOLO11m fine-tuned | effetto della taglia del modello |
| RT-DETR-L fine-tuned | architettura transformer vs CNN |
| Checkpoint Roboflow `basketball-player-detection-3-ycjdo/4` (opzionale) | il modello usato finora nella pipeline |

**Valutazione**
1. *In-domain*: test split del dataset Roboflow (mai visto in training dai modelli che addestriamo qui → risolve il problema dell'indipendenza).
2. *Out-of-domain*: frame di basket di **SportsMOT** (val), dataset indipendente → misura la generalizzazione.

Metriche: AP50-95, AP50, Precision / Recall alla soglia operativa, FPS. Tutto viene salvato su Drive: se Colab si disconnette, rieseguendo il notebook i modelli già addestrati vengono saltati.

Runtime: **Colab con GPU (T4)**.

In [ ]:
%pip install -q ultralytics roboflow pycocotools trackers

In [ ]:
# ==== SETUP COMUNE (uguale in tutti i notebook) ====
import sys, os
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/NBA_Tracking_Project')   # progetto su Drive
    FAST = Path('/content/fast')                                 # disco locale Colab (veloce)
else:
    ROOT = Path(r'F:\Tesi Basket')                               # progetto in locale
    FAST = ROOT / '_cache'
FAST.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Cartella del progetto non trovata: {ROOT}'
print('Progetto:', ROOT)

import json, time, shutil, yaml
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

EXP = ROOT / 'experiments' / 'E1_detection'
EXP.mkdir(parents=True, exist_ok=True)
DATA_DIR = FAST / 'datasets'          # dataset su disco locale (molto più veloce di Drive)
DATA_DIR.mkdir(exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NESSUNA — attivala!')

## 1. Configurazione

Controlla workspace / progetto / versione dal pulsante **Download Dataset → YOLOv11** della pagina Roboflow Universe del dataset che avevi già usato (`basketball_player_detection_v18`).

In [ ]:
RF_WORKSPACE = 'roboflow-jvuqo'                         # <-- VERIFICA
RF_PROJECT   = 'basketball-player-detection-3-ycjdo'    # <-- VERIFICA
RF_VERSION   = 18                                       # <-- VERIFICA

EPOCHS   = 60
IMGSZ    = 640
PATIENCE = 15
SEED     = 0

# nome -> (pesi di partenza, batch). Commenta le righe che non vuoi addestrare.
MODELS_TO_TRAIN = {
    'yolo11n_ft':  ('yolo11n.pt',  16),
    'yolo11s_ft':  ('yolo11s.pt',  16),
    'yolo11m_ft':  ('yolo11m.pt',  8),
    'rtdetr_l_ft': ('rtdetr-l.pt', 4),
}
ZERO_SHOT = {'yolo11s_coco': 'yolo11s.pt'}   # nessun training: classe COCO 'person'

PLAYER_CLASSES = {'player', 'player-in-possession', 'player-jump-shot',
                  'player-layup-dunk', 'player-shot-block', 'person'}
OPERATING_CONF = 0.4

## 2. Download del dataset (formato YOLO)

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ds_path = DATA_DIR / f'{RF_PROJECT}-{RF_VERSION}'
if not ds_path.exists():
    rf = Roboflow(api_key=getpass('API key Roboflow: ').strip())
    rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download('yolov11', location=str(ds_path))

# I data.yaml di Roboflow hanno percorsi relativi che spesso si rompono: li riscriviamo assoluti.
cfg = yaml.safe_load((ds_path / 'data.yaml').read_text())
NAMES = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]
splits = {s: ds_path / s / 'images' for s in ['train', 'valid', 'test']}
for s, p in splits.items():
    assert p.is_dir(), f'Split mancante: {p}'
    print(f'{s:6s}: {len(list(p.glob("*"))):5d} immagini')
DATA_YAML = ds_path / 'data_abs.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path': str(ds_path), 'train': str(splits['train']),
                                     'val': str(splits['valid']), 'test': str(splits['test']),
                                     'nc': len(NAMES), 'names': NAMES}))
print('Classi:', NAMES)

## 3. Training

Ogni modello viene salvato in `experiments/E1_detection/runs/<nome>`. Se `best.pt` esiste già, il training viene saltato.
Tempi indicativi su T4 (dipendono dalla dimensione del dataset): n ≈ 20-30 min, s ≈ 30-45 min, m ≈ 1 h, RT-DETR ≈ 1.5 h.

In [ ]:
from ultralytics import YOLO, RTDETR

def load_model(weights):
    return RTDETR(weights) if 'rtdetr' in str(weights) else YOLO(weights)

RUNS = EXP / 'runs'
WEIGHTS = {}
for name, (init, batch) in MODELS_TO_TRAIN.items():
    best = RUNS / name / 'weights' / 'best.pt'
    if best.is_file():
        print(f'[skip] {name} già addestrato'); WEIGHTS[name] = best; continue
    last = RUNS / name / 'weights' / 'last.pt'
    if last.is_file():                                   # ripresa dopo disconnessione
        load_model(last).train(resume=True)
    else:
        load_model(init).train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=batch,
                               patience=PATIENCE, seed=SEED, project=str(RUNS), name=name,
                               exist_ok=True, plots=True, verbose=False)
    WEIGHTS[name] = best
for name, w in ZERO_SHOT.items():
    WEIGHTS[name] = w
WEIGHTS

## 4. Protocollo di valutazione comune

Stesso protocollo del notebook 02: tutte le classi "giocatore" vengono unificate in una sola classe `player`, valutazione COCO con `pycocotools`, NMS class-agnostic, conf minima 0.001 per l'AP e precision/recall a `OPERATING_CONF` con IoU 0.5.

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import contextlib, io

def yolo_split_to_coco(img_dir, names, keep):
    # GT COCO (solo giocatori, classe unica) dalle label YOLO di uno split.
    lbl_dir = img_dir.parent / 'labels'
    images, anns = [], []
    for i, p in enumerate(sorted(img_dir.glob('*'))):
        h, w = cv2.imread(str(p)).shape[:2]
        images.append({'id': i, 'file_name': str(p), 'width': w, 'height': h})
        lf = lbl_dir / (p.stem + '.txt')
        if not lf.is_file(): continue
        for line in lf.read_text().split('\n'):
            v = line.split()
            if len(v) != 5 or names[int(v[0])] not in keep: continue   # salta anche i poligoni
            cx, cy, bw, bh = map(float, v[1:])
            anns.append({'id': len(anns) + 1, 'image_id': i, 'category_id': 1, 'iscrowd': 0,
                         'bbox': [(cx - bw/2)*w, (cy - bh/2)*h, bw*w, bh*h], 'area': bw*w*bh*h})
    return {'images': images, 'annotations': anns, 'categories': [{'id': 1, 'name': 'player'}]}

def to_coco_obj(d):
    c = COCO(); c.dataset = d
    with contextlib.redirect_stdout(io.StringIO()): c.createIndex()
    return c

def predict(model, gt, keep=PLAYER_CLASSES, imgsz=IMGSZ):
    # Predizioni in formato COCO + FPS misurato (batch 1, dopo warm-up).
    names = model.names
    preds, times = [], []
    imgs = gt['images']
    for _ in range(3): model.predict(imgs[0]['file_name'], imgsz=imgsz, conf=0.001, verbose=False)
    for im in tqdm(imgs, leave=False):
        frame = cv2.imread(im['file_name'])
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter()
        r = model.predict(frame, imgsz=imgsz, conf=0.001, iou=0.7, agnostic_nms=True,
                          max_det=300, verbose=False)[0]
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
        for (x1, y1, x2, y2), s, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(),
                                          r.boxes.cls.cpu().numpy().astype(int)):
            if names[c] in keep:
                preds.append({'image_id': im['id'], 'category_id': 1, 'score': float(s),
                              'bbox': [float(x1), float(y1), float(x2-x1), float(y2-y1)]})
    return preds, 1.0 / np.mean(times)

def coco_metrics(gt, preds, conf=OPERATING_CONF):
    cg = to_coco_obj(gt)
    with contextlib.redirect_stdout(io.StringIO()):
        cd = cg.loadRes(preds) if preds else to_coco_obj({**gt, 'annotations': []})
        ev = COCOeval(cg, cd, 'bbox'); ev.evaluate(); ev.accumulate(); ev.summarize()
        op = COCOeval(cg, cd, 'bbox')
        op.params.iouThrs = np.array([0.5]); op.params.areaRng = [[0, 1e10]]
        op.params.areaRngLbl = ['all']; op.params.maxDets = [300]; op.evaluate()
    tp = fp = fn = 0
    for e in op.evalImgs:
        if e is None: continue
        sel = (np.asarray(e['dtScores']) >= conf) & ~np.asarray(e['dtIgnore'][0], bool)
        m = e['dtMatches'][0]
        tp += int(np.sum(sel & (m > 0))); fp += int(np.sum(sel & (m == 0)))
        matched = set(m[sel]) - {0}
        fn += sum((not ig) and g not in matched for g, ig in zip(e['gtIds'], e['gtIgnore']))
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    pr = ev.eval['precision'][0, :, 0, 0, -1]          # curva PR @ IoU 0.5
    return {'AP50_95': ev.stats[0], 'AP50': ev.stats[1], 'AP75': ev.stats[2],
            'Precision': P, 'Recall': R, 'F1': 2*P*R / max(P + R, 1e-9),
            'TP': tp, 'FP': fp, 'FN': fn}, pr

## 5. Valutazione in-domain (test split Roboflow)

In [ ]:
GT_TEST = yolo_split_to_coco(splits['test'], NAMES, PLAYER_CLASSES)
print('Immagini test:', len(GT_TEST['images']), '| giocatori annotati:', len(GT_TEST['annotations']))

rows, PR_CURVES = [], {}
for name, w in WEIGHTS.items():
    model = load_model(w)
    preds, fps = predict(model, GT_TEST)
    m, pr = coco_metrics(GT_TEST, preds)
    n_params = sum(p.numel() for p in model.model.parameters()) / 1e6
    rows.append({'model': name, **m, 'FPS_T4': fps, 'params_M': n_params})
    PR_CURVES[name] = pr
    json.dump(preds, open(EXP / f'preds_test_{name}.json', 'w'))
    print(f'{name:14s} AP50-95={m["AP50_95"]:.3f}  AP50={m["AP50"]:.3f}  P={m["Precision"]:.3f}  R={m["Recall"]:.3f}  FPS={fps:.1f}')

res_in = pd.DataFrame(rows).set_index('model').round(4)
res_in.to_csv(EXP / 'results_in_domain.csv')
res_in

### (Opzionale) checkpoint Roboflow usato finora
Il checkpoint `basketball-player-detection-3-ycjdo/4` è servito via `inference`: lo valutiamo sulle **stesse** immagini. Attenzione: non sappiamo se è stato addestrato su queste immagini di test → nella tesi va riportato separatamente, con questa nota.

In [ ]:
RUN_ROBOFLOW = False
if RUN_ROBOFLOW:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'inference-gpu'], check=True)
    from inference import get_model
    from getpass import getpass
    rf_model = get_model('basketball-player-detection-3-ycjdo/4', api_key=getpass('API key: '))
    preds, times = [], []
    for im in tqdm(GT_TEST['images']):
        f = cv2.imread(im['file_name']); t0 = time.perf_counter()
        r = rf_model.infer(f, confidence=0.001, iou_threshold=0.7)[0]; times.append(time.perf_counter() - t0)
        for p in r.predictions:
            if p.class_name in PLAYER_CLASSES:
                preds.append({'image_id': im['id'], 'category_id': 1, 'score': float(p.confidence),
                              'bbox': [p.x - p.width/2, p.y - p.height/2, p.width, p.height]})
    m, pr = coco_metrics(GT_TEST, preds)
    res_in.loc['roboflow_yolo11s*'] = {**m, 'FPS_T4': 1/np.mean(times), 'params_M': np.nan}
    PR_CURVES['roboflow_yolo11s*'] = pr
    res_in.round(4).to_csv(EXP / 'results_in_domain.csv')
res_in

## 6. Valutazione out-of-domain: SportsMOT (basket, split val)

Il download avviene in automatico con la libreria `trackers` di Roboflow (lo zip resta su Drive in `datasets/_cache_sportsmot`, così non si riscarica). La lista delle sequenze di basket viene scaricata da Hugging Face; se il download fallisce, il messaggio spiega come fare a mano.

Campioniamo 1 frame ogni `STEP`. In SportsMOT sono annotati **solo i giocatori in campo** (non arbitri e pubblico): un detector che non li distingue viene penalizzato in precision, ed è proprio ciò che vogliamo misurare.

In [ ]:
# ==== SportsMOT: download (val), ricerca sequenze e lista basket ====
import subprocess, urllib.request, configparser

SMOT_DIR = FAST / 'SportsMOT'
SMOT_CACHE = ROOT / 'datasets' / '_cache_sportsmot'     # zip conservati su Drive: niente ri-download
BASKET_LIST = ROOT / 'datasets' / 'sportsmot_basketball.txt'

def sportsmot_val_sequences():
    if not SMOT_DIR.exists() or not any(SMOT_DIR.rglob('gt.txt')):
        SMOT_CACHE.mkdir(parents=True, exist_ok=True)
        subprocess.run(['trackers', 'download', 'sportsmot', '--split', 'val',
                        '--output', str(SMOT_DIR), '--cache-dir', str(SMOT_CACHE)], check=True)
    seqs = sorted({p.parent.parent for p in SMOT_DIR.rglob('gt/gt.txt')})
    assert seqs, f'Nessuna sequenza trovata in {SMOT_DIR}'
    return seqs

def basketball_names():
    if not BASKET_LIST.is_file():
        url = 'https://huggingface.co/datasets/MCG-NJU/SportsMOT/resolve/main/splits_txt/basketball.txt'
        try:
            BASKET_LIST.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, BASKET_LIST)
        except Exception as e:
            raise RuntimeError(
                f'Non riesco a scaricare la lista basket ({e}).\n'
                f'Scaricala a mano da {url} (serve login Hugging Face) e salvala in {BASKET_LIST}')
    return set(BASKET_LIST.read_text().split())

def seq_info(seq):
    ini = seq / 'seqinfo.ini'
    fps, n = 25.0, len(list((seq / 'img1').glob('*.jpg')))
    if ini.is_file():
        c = configparser.ConfigParser(); c.read(ini)
        fps = float(c['Sequence'].get('frameRate', fps))
    return fps, n

STEP = 10
RUN_SPORTSMOT = True

def sportsmot_to_coco(seqs, step=STEP):
    images, anns = [], []
    for seq in seqs:
        gt = np.loadtxt(seq / 'gt' / 'gt.txt', delimiter=',', ndmin=2)
        frames = sorted((seq / 'img1').glob('*.jpg'))
        h, w = cv2.imread(str(frames[0])).shape[:2]
        for fi in range(0, len(frames), step):
            iid = len(images)
            images.append({'id': iid, 'file_name': str(frames[fi]), 'width': w, 'height': h})
            for r in gt[gt[:, 0] == fi + 1]:                  # frame MOT 1-based
                anns.append({'id': len(anns)+1, 'image_id': iid, 'category_id': 1, 'iscrowd': 0,
                             'bbox': [float(r[2]), float(r[3]), float(r[4]), float(r[5])], 'area': float(r[4]*r[5])})
    return {'images': images, 'annotations': anns, 'categories': [{'id': 1, 'name': 'player'}]}

if RUN_SPORTSMOT:
    basket = basketball_names()
    seqs = [s for s in sportsmot_val_sequences() if s.name in basket]
    GT_SMOT = sportsmot_to_coco(seqs)
    print(f'Sequenze basket val: {len(seqs)} | frame campionati: {len(GT_SMOT["images"])} | box: {len(GT_SMOT["annotations"])}')
    rows = []
    for name, w in WEIGHTS.items():
        preds, fps = predict(load_model(w), GT_SMOT)
        m, _ = coco_metrics(GT_SMOT, preds)
        rows.append({'model': name, **m})
        print(f'{name:14s} AP50-95={m["AP50_95"]:.3f}  AP50={m["AP50"]:.3f}  P={m["Precision"]:.3f}  R={m["Recall"]:.3f}')
    res_out = pd.DataFrame(rows).set_index('model').round(4)
    res_out.to_csv(EXP / 'results_sportsmot.csv')
    display(res_out)

## 7. Grafici per la tesi

In [ ]:
FIG = EXP / 'figures'; FIG.mkdir(exist_ok=True)
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

# (a) AP in-domain vs out-of-domain
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(res_in)); wd = 0.38
ax.bar(x - wd/2, res_in['AP50_95'], wd, label='Roboflow test (in-domain)', color='#2a6f97')
if 'res_out' in globals():
    ax.bar(x + wd/2, res_out.reindex(res_in.index)['AP50_95'], wd, label='SportsMOT (out-of-domain)', color='#e07a5f')
ax.set_xticks(x, res_in.index, rotation=20); ax.set_ylabel('AP50-95'); ax.set_ylim(0, 1); ax.legend()
ax.set_title('Detection giocatori: accuratezza per modello'); fig.tight_layout(); fig.savefig(FIG / 'ap_per_modello.png', dpi=200)

# (b) trade-off velocità / accuratezza
fig, ax = plt.subplots(figsize=(6, 4.5))
for n, r in res_in.iterrows():
    ax.scatter(r['FPS_T4'], r['AP50_95'], s=60); ax.annotate(n, (r['FPS_T4'], r['AP50_95']), xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('FPS (T4, batch 1)'); ax.set_ylabel('AP50-95'); ax.set_title('Trade-off velocità / accuratezza')
fig.tight_layout(); fig.savefig(FIG / 'fps_vs_ap.png', dpi=200)

# (c) curve precision-recall @ IoU 0.5
fig, ax = plt.subplots(figsize=(6, 4.5)); rec = np.linspace(0, 1, 101)
for n, pr in PR_CURVES.items():
    ax.plot(rec[pr > -1], pr[pr > -1], label=n)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_title('Curve PR (IoU 0.5) — test in-domain'); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(FIG / 'pr_curves.png', dpi=200)
plt.show()

## 8. Confronto qualitativo sullo stesso frame

In [ ]:
def draw(model, path, conf=OPERATING_CONF):
    img = cv2.imread(path); r = model.predict(img, conf=conf, imgsz=IMGSZ, agnostic_nms=True, verbose=False)[0]
    for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.cpu().numpy().astype(int), r.boxes.cls.cpu().numpy().astype(int)):
        if model.names[c] in PLAYER_CLASSES: cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

sample_paths = [GT_TEST['images'][len(GT_TEST['images']) // 2]['file_name']]
if 'GT_SMOT' in globals(): sample_paths.append(GT_SMOT['images'][len(GT_SMOT['images']) // 2]['file_name'])
models = {n: load_model(w) for n, w in WEIGHTS.items()}
fig, axes = plt.subplots(len(sample_paths), len(models), figsize=(4.5 * len(models), 3 * len(sample_paths)), squeeze=False)
for i, p in enumerate(sample_paths):
    for j, (n, m) in enumerate(models.items()):
        axes[i, j].imshow(draw(m, p)); axes[i, j].axis('off')
        if i == 0: axes[i, j].set_title(n)
fig.tight_layout(); fig.savefig(FIG / 'qualitativo.png', dpi=150); plt.show()
print('Tutto salvato in', EXP)

## 9. Scelta del detector per il tracking

Il detector usato negli esperimenti E2 e nella demo viene scelto **in base all'AP su SportsMOT** (generalizzazione) o, se SportsMOT non è stato eseguito, sul test Roboflow. Viene copiato in `models/detector_best.pt`, e anche il COCO pre-addestrato viene salvato come riferimento.

In [ ]:
table = res_out if 'res_out' in globals() else res_in
candidates = table.drop(index=[k for k in table.index if k not in MODELS_TO_TRAIN], errors='ignore')
best_name = candidates['AP50_95'].idxmax()
(ROOT / 'models').mkdir(exist_ok=True)
shutil.copy2(WEIGHTS[best_name], ROOT / 'models' / 'detector_best.pt')
(ROOT / 'models' / 'detector_best.txt').write_text(best_name)
for n in MODELS_TO_TRAIN:
    if Path(WEIGHTS[n]).is_file(): shutil.copy2(WEIGHTS[n], ROOT / 'models' / f'{n}.pt')
print('Detector scelto:', best_name, '-> models/detector_best.pt')